In [2]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import gensim
from gensim import corpora
from gensim.models import CoherenceModel, Phrases
from gensim.models.phrases import Phraser

import string
from pprint import pprint

import pyLDAvis
import pyLDAvis.gensim_models
import matplotlib.pyplot as plt
%matplotlib inline

import pandas as pd

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\glenv\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\glenv\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
df = pd.read_csv('240592T_data.csv')

df.head()


,clean_text,document_type
0,public information awareness service vulnerabl...,APPROVAL
1,report project completion report panama colon ...,APPROVAL
2,document file copy report uni report recommend...,APPROVAL
3,report china ningbo shanghai port project ning...,REVIEW
4,independent evaluation group ieg icr review ke...,REVIEW


In [ ]:
df_tokenized = df["clean_text"].dropna().apply(word_tokenize).tolist()

dictionary = corpora.Dictionary(df_tokenized)

doc_term_matrix = [dictionary.doc2bow(doc) for doc in df_tokenized]

topic_num =15
word_num = 5

Lda = gensim.models.ldamodel.LdaModel
ldamodel = Lda(doc_term_matrix, num_topics=topic_num, id2word=dictionary, random_state=42, passes=20)
pprint(ldamodel.print_topics(num_topics=topic_num, num_words=word_num))



[(0,
  '0.019*"project" + 0.011*"power" + 0.011*"energy" + 0.007*"cost" + '
  '0.007*"financial"'),
 (1,
  '0.017*"loan" + 0.013*"bank" + 0.012*"million" + 0.009*"would" + '
  '0.009*"project"'),
 (2, '0.029*"de" + 0.016*"le" + 0.009*"project" + 0.008*"pour" + 0.006*"dans"'),
 (3,
  '0.031*"project" + 0.008*"cost" + 0.007*"would" + 0.006*"bank" + '
  '0.006*"year"'),
 (4,
  '0.010*"del" + 0.008*"project" + 0.008*"los" + 0.007*"municipality" + '
  '0.006*"state"'),
 (5,
  '0.023*"project" + 0.020*"education" + 0.017*"school" + 0.012*"training" + '
  '0.008*"teacher"'),
 (6,
  '0.064*"health" + 0.014*"care" + 0.012*"theâ" + 0.012*"hiv" + '
  '0.011*"service"'),
 (7,
  '0.013*"sector" + 0.012*"program" + 0.011*"government" + 0.010*"reform" + '
  '0.010*"public"'),
 (8,
  '0.022*"project" + 0.011*"management" + 0.007*"area" + 0.007*"activity" + '
  '0.007*"development"'),
 (9,
  '0.034*"project" + 0.009*"implementation" + 0.007*"component" + 0.007*"icr" '
  '+ 0.006*"indicator"')]


### Baseline Model Parameters

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `num_topics` | 10 | A moderate starting point for exploratory analysis. Starting with 10 topics provides enough granularity to observe whether the corpus contains distinct themes, while being small enough to allow manual inspection of each topic's word distribution. |
| `num_words` | 5 | Displaying 5 words per topic provides enough context to interpret each topic's theme without introducing noise from low-weight terms. |
| `passes` | 20 | Controls how many times the algorithm iterates over the entire corpus. 20 passes gives the model sufficient opportunity to converge, as too few passes (e.g., 1–5) may result in unstable topic-word distributions. |
| `random_state` | 42 | Ensures reproducibility so that results are consistent across runs. |

This baseline serves as a reference point. The topics produced will be examined qualitatively using pyLDAvis and quantitatively using coherence and perplexity metrics to determine whether the topic count should be adjusted.

In [11]:
perplexity = ldamodel.log_perplexity(doc_term_matrix)
print(f'Perplexity: {perplexity:.4f}')

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(ldamodel, doc_term_matrix, dictionary)
vis 

Perplexity: -8.3481


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
9     -0.065964 -0.016502       1        1  22.891320
3     -0.090056 -0.009112       2        1  21.819189
8     -0.039055 -0.040616       3        1  16.731585
7     -0.058197 -0.024500       4        1  11.282452
5     -0.026966 -0.033287       5        1   7.978099
0     -0.075718 -0.035657       6        1   6.746884
1     -0.085457 -0.034528       7        1   6.376505
4      0.024072  0.103023       8        1   2.445208
6      0.297121 -0.140817       9        1   1.947557
2      0.120219  0.231996      10        1   1.781201, topic_info=                Term           Freq          Total Category  logprob  loglift
273           health   32030.000000   32030.000000  Default  30.0000  30.0000
178        education   30924.000000   30924.000000  Default  29.0000  29.0000
509          project  383497.000000  383497.000000  Default  28.0000  28.0000
567           school   25529.000000   25529.000000  Default  27.0000  27.0000
1338            loan   49058.000000   49058.000000  Default  26.0000  26.0000
...              ...            ...            ...      ...      ...      ...
158      development     723.497351   91496.444306  Topic10  -6.0388  -0.8121
1481     performance     681.720153   44988.890894  Topic10  -6.0983  -0.1617
971           credit     606.145849   32877.791383  Topic10  -6.2158   0.0345
289   implementation     617.544909   83217.932122  Topic10  -6.1971  -0.8756
420         national     606.513638   37817.601226  Topic10  -6.2152  -0.1049

[914 rows x 6 columns], token_table=        Topic      Freq      Term
term                             
353513      5  0.999023       2ie
63876       1  0.005485  abattoir
63876       2  0.990095  abattoir
4914        1  0.026068  academic
4914        2  0.032150  academic
...       ...       ...       ...
6742        7  0.038860         ï
6742        9  0.003048         ï
6742       10  0.011212         ï
34027       2  0.039715       ï¿½
34027       9  0.959434       ï¿½

[3430 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[10, 4, 9, 8, 6, 1, 2, 5, 7, 3])

### Observations from Baseline Model

The pyLDAvis intertopic distance map of the 10-topic baseline model revealed significant overlap among topics 1 through 7, which clustered tightly in the same quadrant. This indicates that these topics share substantial vocabulary and are not capturing distinct themes. Only topics 8, 9, and 10 showed clear spatial separation, suggesting the corpus may contain fewer than 10 genuinely distinct topics.

The marginal topic distribution further confirmed this: most topics accounted for only 2–5% of the corpus, with no clear differentiation in size among the overlapping cluster.

Based on these observations, the number of topics was reduced for the hyperparameter search to focus on the range [3, 5, 7], which better reflects the likely number of distinct themes in the data.

**1. Bigram Phrases**

Bigrams were introduced using Gensim's `Phrases` model to capture meaningful multi-word expressions (e.g., "climate_change", "human_rights") that would otherwise be split into separate, less informative unigrams.

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `min_count` | 5 | A bigram must appear at least 5 times to be retained, filtering out rare co-occurrences that are likely noise. |
| `threshold` | 100 | A higher threshold is more conservative, only forming bigrams where two words co-occur significantly more than expected by chance. A value of 100 balances between capturing meaningful phrases and avoiding spurious ones. |

**2. Dictionary Filtering**

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `no_below` | 5 | Removes words appearing in fewer than 5 documents. Very rare words add noise and do not contribute to coherent topics. |
| `no_above` | 0.7 | Removes words appearing in more than 70% of documents. These overly common words do not discriminate between topics. |
| `keep_n` | 20000 | Caps the vocabulary at 20,000 tokens to keep the model computationally efficient while retaining the most informative terms. |

**3. Hyperparameter Grid Search**

| Parameter | Values Tested | Rationale |
|-----------|--------------|-----------|
| `num_topics` | [3, 5, 7] | The baseline model with 10 topics showed heavy overlap among 7 of the 10 topics in the pyLDAvis intertopic distance map, while only 3 topics were clearly distinct. This suggests the corpus contains between 3 and 7 meaningful themes. Testing this range allows systematic identification of the optimal granularity. |
| `passes` | [10, 20] | Testing whether fewer passes achieve comparable convergence, or whether 20 passes remains necessary for stable topic distributions. |
| `iterations` | 400 | Increased from the default (50) to give the variational inference E-step more iterations per document, improving convergence particularly for models with fewer topics where each topic must capture broader themes. |
| `chunksize` | 2000 | A larger chunk size reduces stochastic noise during online variational Bayes training, producing more stable parameter estimates. |

Each combination is evaluated using **c_v coherence** (measures semantic similarity of top words within each topic; higher is better) and **log perplexity** (measures how well the model predicts the corpus; closer to 0 is better).

In [ ]:
bigram_model = Phrases(df_tokenized, min_count=5, threshold=100)
bigram_phraser = Phraser(bigram_model)
df_with_bigrams = [bigram_phraser[doc] for doc in df_tokenized]

topic_num = [3, 5]
passes_list = [10, 20]

results = []

dictionary = corpora.Dictionary(df_with_bigrams)
dictionary.filter_extremes(no_above=0.7, no_below=5, keep_n=20000)
doc_term_matrix = [dictionary.doc2bow(doc) for doc in df_with_bigrams]

for k in topic_num:
    for p in passes_list:
        ldamodel = Lda(
            corpus=doc_term_matrix,
            num_topics=k,
            id2word=dictionary,
            random_state=42,
            passes=p,
            iterations=400,  
            chunksize=2000,
        )

        log_perplexity = ldamodel.log_perplexity(doc_term_matrix)  # log perplexity (closer to 0 is better.)
        perplexity = 2 ** (-log_perplexity) 

        coherence_model = CoherenceModel(
            model=ldamodel,
            texts=df_with_bigrams,
            dictionary=dictionary,
            coherence="c_v"
        )
        coherence = coherence_model.get_coherence()

        results.append((k, p, log_perplexity, perplexity, coherence))
        print(
            f"Topics={k} | passes={p} | log_perplexity={log_perplexity:.4f} "
            f"| perplexity={perplexity:.4f} | c_v coherence={coherence:.4f}"
        )

results_df = pd.DataFrame(
    results,
    columns=["num_topics", "passes", "log_perplexity", "perplexity", "c_v_coherence"]
).sort_values(["c_v_coherence", "log_perplexity"], ascending=[False, False])

results_df

Topics= 3 | passes=10 | log_perplexity=-8.1039 | perplexity=275.1091 | c_v coherence=0.3619
Topics= 3 | passes=20 | log_perplexity=-8.1021 | perplexity=274.7775 | c_v coherence=0.3680
Topics= 5 | passes=10 | log_perplexity=-8.0481 | perplexity=264.6790 | c_v coherence=0.4107
Topics= 5 | passes=20 | log_perplexity=-8.0449 | perplexity=264.0990 | c_v coherence=0.4129


,num_topics,passes,log_perplexity,perplexity,c_v_coherence
3,5,20,-8.044935,264.099012,0.412880
2,5,10,-8.048100,264.678966,0.410674
1,3,20,-8.102120,274.777487,0.367954
0,3,10,-8.103860,275.109074,0.361915


### Grid Search Results Analysis

The grid search evaluated 6 configurations across 3 topic numbers [3, 5, 7] and 2 pass values [10, 20], using c_v coherence and perplexity as evaluation metrics.

**Key observations:**

The results show a consistent pattern: both coherence and perplexity improve as the number of topics increases from 3 to 7. The 7-topic models achieved the highest coherence scores (0.5159 and 0.5331) and the lowest perplexity values (257.10 and 256.10), while the 3-topic models performed worst on both metrics (coherence ~0.36, perplexity ~275).

Increasing passes from 10 to 20 produced marginal improvements across all topic numbers. For example, with 7 topics, coherence improved from 0.5159 to 0.5331 (+0.017) and perplexity decreased from 257.10 to 256.10. This suggests that 10 passes achieves near-convergence, but 20 passes provides a small additional benefit at the cost of longer training time.

**Selected model: 7 topics, 20 passes**

This configuration was selected as the best-performing model because it achieved the highest c_v coherence (0.5331) and the lowest perplexity (256.10) among all configurations tested. Both metrics agree on this selection, which strengthens confidence in the result.

A c_v coherence of 0.53 indicates moderate topic coherence. While not exceptionally high, this is reasonable for a general-purpose corpus covering world issues, where topics may naturally share some vocabulary across themes such as governance, finance, and development.

| Metric | Best Model (k=7, passes=20) | Worst Model (k=3, passes=10) | Improvement |
|--------|----------------------------|------------------------------|-------------|
| c_v Coherence | 0.5331 | 0.3620 | +47.3% |
| Perplexity | 256.10 | 275.11 | -6.9% (lower is better) |
| Log Perplexity | -8.0006 | -8.1039 | Closer to 0 |

Out of all the parameters I have tested, the model performs best with 7 topics and 20 passes, so I will do another round of tuning where i try to narrow down the exact number of topics between 6 and 9 and find out which set of parameters makes the model performs best.


In [17]:
topic_num = [6, 7, 8, 9]
passes = 20

second_round_results = []

for k in topic_num:
    ldamodel = Lda(
        corpus=doc_term_matrix, 
        num_topics=k, 
        id2word=dictionary, 
        random_state=42, 
        passes=passes,
        iterations=400, 
        chunksize=2000
    )

    log_perplexity = ldamodel.log_perplexity(doc_term_matrix)

    perplexity = 2 ** (-log_perplexity)

    coherence_model = CoherenceModel(model=ldamodel, texts = df_with_bigrams, 
                           dictionary=dictionary, coherence="c_v")
    
    coherence = coherence_model.get_coherence()

    second_round_results.append((k, p, log_perplexity, perplexity, coherence))
    print(
        f"Topics={k} | passes={passes} | log_perplexity={log_perplexity:.4f} "
        f"| perplexity={perplexity:.4f} | c_v coherence={coherence:.4f}"
    )

second_round_results_df = pd.DataFrame(
    second_round_results,
    columns=["num_topics", "passes", "log_perplexity", "perplexity", "c_v_coherence"]
).sort_values(["c_v_coherence", "log_perplexity"], ascending=[False, False])

second_round_results_df

Topics=6 | passes=20 | log_perplexity=-8.0226 | perplexity=260.0368 | c_v coherence=0.4582
Topics=7 | passes=20 | log_perplexity=-8.0006 | perplexity=256.1041 | c_v coherence=0.5331
Topics=8 | passes=20 | log_perplexity=-7.9818 | perplexity=252.7859 | c_v coherence=0.5172
Topics=9 | passes=20 | log_perplexity=-7.9709 | perplexity=250.8952 | c_v coherence=0.4967


,num_topics,passes,log_perplexity,perplexity,c_v_coherence
1,7,20,-8.000586,256.104075,0.533112
2,8,20,-7.981772,252.785855,0.517234
3,9,20,-7.970941,250.895204,0.496677
0,6,20,-8.022572,260.036791,0.458199


### Improvement Attempt 2: Fine-Grained Topic Number Confirmation

The initial grid search tested [3, 5, 7] topics and found that coherence increased consistently up to k=7, which was the upper bound of the tested range. Since 7 was at the boundary, it is necessary to verify whether coherence peaks at exactly 7 or continues rising slightly beyond it.

However, the baseline 10-topic model already demonstrated heavy overlap among 7 of the 10 topics in the pyLDAvis intertopic distance map. Therefore, testing topic numbers significantly above 9 is not justified — the evidence already shows that the corpus does not support that many distinct themes. A targeted search over [6, 7, 8, 9] was conducted to pinpoint the exact peak.

**Results:**

| Topics | Perplexity | Log Perplexity | c_v Coherence |
|--------|-----------|----------------|---------------|
| 6 | 260.04 | -8.0226 | 0.4582 |
| 7 | 256.10 | -8.0006 | 0.5331 |
| 8 | 252.79 | -7.9818 | 0.5172 |
| 9 | 250.90 | -7.9709 | 0.4967 |

**Observations:**

The c_v coherence follows a clear inverted-U pattern, rising from 0.4582 at k=6 to a peak of 0.5331 at k=7, then declining to 0.5172 at k=8 and 0.4967 at k=9. This confirms that 7 topics is the optimal number — not merely the best within the initial [3, 5, 7] range, but a genuine peak with lower coherence on both sides. The drop from k=7 to k=9 represents a 6.8% decline in coherence, indicating a meaningful deterioration in topic quality rather than random variation.

The perplexity decreases monotonically as topic numbers increase (260.04 → 256.10 → 252.79 → 250.90). This is expected behaviour because adding more topics always gives the model greater flexibility to fit the training data. However, perplexity alone is not a reliable criterion for selecting the number of topics, as it does not account for topic interpretability — it will always favour more topics regardless of whether they are meaningful. The coherence metric, which measures how semantically related the top words within each topic are, provides a more appropriate guide and clearly favours k=7.

The decline in coherence from k=7 to k=8 and k=9 is consistent with the baseline observation: when forced to produce more topics than the corpus naturally supports, the model begins splitting coherent themes into sub-topics with overlapping vocabulary, reducing overall coherence.

**Conclusion:** k=7 is confirmed as the optimal number of topics and will be carried forward to the next improvement attempt, which will tune the alpha and eta hyperparameters.

In [23]:
best_k = 7 

alpha_eta_configs = [
    ("symmetric", "symmetric"),
    ("symmetric", "auto"),
    ("auto", "symmetric"),
    ("auto", "auto"),
    ("asymmetric", "symmetric")
]

alpha_eta_results = []

for alpha_val, eta_val in alpha_eta_configs:
    ldamodel = Lda(
        corpus=doc_term_matrix,
        num_topics=best_k,
        id2word=dictionary, 
        random_state=42,
        passes=20,
        iterations=400,
        chunksize=2000, 
        alpha=alpha_val, 
        eta=eta_val
    )

    log_perplexity = ldamodel.log_perplexity(doc_term_matrix)
    perplexity = 2 ** (-log_perplexity)

    coherence_model = CoherenceModel(model=ldamodel, texts=df_with_bigrams,
                                     dictionary=dictionary, coherence="c_v")
    
    coherence = coherence_model.get_coherence()

    alpha_eta_results.append((alpha_val, eta_val, log_perplexity, perplexity, coherence))

    print(
        f"Alpha={alpha_val} | eta={eta_val} | log_perplexity={log_perplexity:.4f} "
        f"| perplexity={perplexity:.4f} | c_v coherence={coherence:.4f}"
    )

alpha_eta_results_df = pd.DataFrame(
    alpha_eta_results,
    columns=["alpha_val", "eta_val", "log_perplexity", "perplexity", "c_v_coherence"]
).sort_values(["c_v_coherence", "log_perplexity"], ascending=[False, False])

alpha_eta_results_df


Alpha=symmetric | eta=symmetric | log_perplexity=-8.0006 | perplexity=256.1041 | c_v coherence=0.5331
Alpha=symmetric | eta=auto | log_perplexity=-7.9960 | perplexity=255.2910 | c_v coherence=0.5331
Alpha=auto | eta=symmetric | log_perplexity=-8.0005 | perplexity=256.0833 | c_v coherence=0.5331
Alpha=auto | eta=auto | log_perplexity=-7.9959 | perplexity=255.2693 | c_v coherence=0.5331
Alpha=asymmetric | eta=symmetric | log_perplexity=-8.0008 | perplexity=256.1342 | c_v coherence=0.5331


,alpha_val,eta_val,log_perplexity,perplexity,c_v_coherence
3,auto,auto,-7.995876,255.269279,0.533112
1,symmetric,auto,-7.995999,255.290959,0.533112
2,auto,symmetric,-8.000469,256.083290,0.533112
0,symmetric,symmetric,-8.000586,256.104075,0.533112
4,asymmetric,symmetric,-8.000756,256.134247,0.533112


### Improvement Attempt 3: Alpha and Eta Hyperparameter Tuning

Previous improvement attempts focused on feature engineering (bigrams, dictionary filtering) and topic number optimisation. This attempt explores a fundamentally different axis of model control — tuning the Dirichlet prior hyperparameters alpha and eta — while keeping the number of topics fixed at k = 7.

Alpha governs the document–topic distribution and determines how many topics are expected to appear within a single document. Eta governs the topic–word distribution and controls how concentrated or diffuse each topic’s vocabulary is. The default symmetric priors assume equal weighting across topics and words, which may not always reflect the true underlying structure of a corpus.

To evaluate whether relaxing these assumptions improves topic quality, five configurations were tested:

| Alpha | Eta | Rationale |
|-------|-----|-----------|
| symmetric | symmetric | Default baseline for comparison |
| symmetric | auto | Learn optimal topic-word distribution from data |
| auto | symmetric | Learn optimal document-topic distribution from data |
| auto | auto | Learn both distributions from data |
| asymmetric | symmetric | Decaying prior allowing some topics to be more prevalent |

**Results:**

| Alpha | Eta | Perplexity | Log Perplexity | c_v Coherence |
|-------|-----|-----------|----------------|---------------|
| symmetric | symmetric | 256.10 | -8.0006 | 0.5331 |
| symmetric | auto | 255.29 | -7.9960 | 0.5331 |
| auto | symmetric | 256.08 | -8.0005 | 0.5331 |
| auto | auto | 255.27 | -7.9959 | 0.5331 |
| asymmetric | symmetric | 256.13 | -8.0008 | 0.5331 |

**Observations:**

All five configurations produced identical c_v coherence scores of 0.5331, despite minor variations in perplexity. While the auto/auto configuration achieved the lowest perplexity, the difference across configurations was less than 1%, indicating that alpha and eta tuning had a negligible impact on overall model fit.

The invariance of the c_v coherence score can be explained by how coherence is computed. c_v coherence is driven primarily by the semantic co-occurrence of the top-ranked words within each topic, rather than by small probability shifts within the topic–word or document–topic distributions. Across all alpha and eta settings, the learned topics converged to nearly identical sets of top words, resulting in unchanged word co-occurrence statistics and, consequently, identical coherence values.

This suggests that the topic structure learned by the model is already stable and well-defined under the default symmetric priors. The corpus appears to exhibit a relatively balanced topic distribution, with no strong skew toward sparse or highly dominant topics. In such cases, allowing the model to automatically learn or skew the priors does not materially alter topic interpretability, even if minor improvements in likelihood (perplexity) are observed.

**Conclusion:**

Alpha and eta hyperparameter tuning did not improve topic coherence beyond the default configuration. Although small differences in perplexity indicate marginal changes in model likelihood, these changes did not translate into more interpretable or semantically coherent topics. As a result, the final model retains the symmetric/symmetric prior configuration for clarity, simplicity, and reproducibility, achieving a c_v coherence score of 0.5331 with k = 7, passes = 20, iterations = 400, and chunksize = 2000.

## Dataset Source and Usage

The dataset used in this assignment is the **raw_review_Video_games** subset of the  
**McAuley-Lab/Amazon-Reviews-2023** dataset.

This dataset is a large-scale academic research corpus. While a traditional open-source
software license (e.g., MIT or Apache) is not specified on the dataset card, its use in an
academic context is governed by the following conditions:

1. **Mandatory Citation**  
   The academic paper that introduced this version of the dataset must be formally cited.

2. **Academic / Research Use Only**  
   The dataset is intended strictly for non-commercial academic and research purposes.
   It is not used for any commercial application in this assignment.

## Dataset Citation

Hou, Y., Li, J., He, Z., Yan, A., Chen, X., & McAuley, J. (2024).  
*Bridging Language and Items for Retrieval and Recommendation*.  
arXiv preprint arXiv:2403.03952.